# Dapr

![](https://docs.dapr.io/images/overview.png)

Quelle: [dapr.io](https://docs.dapr.io/concepts/overview/)

- - -

Installation


In [ ]:
%%bash
helm repo add dapr https://dapr.github.io/helm-charts/
helm upgrade --install dapr dapr/dapr --version=1.16 --namespace dapr-system --create-namespace --wait
helm install dapr-dashboard dapr/dapr-dashboard --namespace dapr-system

Verbindung zum LLM einrichten

* [OpenAI](https://docs.dapr.io/reference/components-reference/supported-conversation/openai/)

In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: dapr.io/v1alpha1
kind: Component
metadata:
  name: openai
  namespace: default
spec:
  type: conversation.openai
  version: v1
  metadata:
  - name: key
    value: "sglan"
  - name: model
    value: "Qwen/Qwen2.5-0.5B-Instruct"
  - name: endpoint
    value: "http://10.3.24.11:30000/v1"
  - name: responseCacheTTL
    value: "10m"
EOF

In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: dapr.io/v1alpha1
kind: Component
metadata:
  name: llm-provider
  namespace: default
spec:
  type: conversation.openai
  version: v1
  metadata:
  - name: key
    value: "sglan"
  - name: model
    value: "Qwen/Qwen2.5-0.5B-Instruct"
  - name: endpoint
    value: "http://10.3.24.11:30000/v1"
  - name: responseCacheTTL
    value: "10m"
EOF

## Jupyter Lab

Lab Umgebung mit SideCar starten. 

Ermöglicht den Zugriff über obige Componente via Dapr auf das LLM

In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: apps/v1
kind: Deployment
metadata:
  name: jupyter-dapr
  namespace: default
spec:
  replicas: 1
  selector:
    matchLabels:
      app: jupyter-dapr
  template:
    metadata:
      labels:
        app: jupyter-dapr
      annotations:
        dapr.io/enabled: "true"
        dapr.io/app-id: "jupyter-dapr"
        dapr.io/app-port: "8888"
        dapr.io/log-level: "debug"
    spec:
      containers:
      - name: notebook
        image: quay.io/jupyter/minimal-notebook:latest
        ports:
        - containerPort: 8888
        env:
        - name: JUPYTER_TOKEN
          value: "changeme"
        command:
          - start-notebook.sh
          - --NotebookApp.token=''
          - --NotebookApp.password=''
          - --NotebookApp.allow_origin='*'
          - --NotebookApp.ip=0.0.0.0
          - --NotebookApp.allow_root=true
EOF


In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: v1
kind: Service
metadata:
  name: jupyter-dapr
  namespace: default
spec:
  selector:
    app: jupyter-dapr
  ports:
  - name: http
    port: 8888
    targetPort: 8888
  type: NodePort
EOF

Zugriff bzw. URL ausga

In [ ]:
%%bash
echo "K8s Dashboard   : https://$(cat ~/work/server-ip):30443"
echo "Dapr Jupyter Lab: http://"$(cat ~/work/server-ip)":$(kubectl get service jupyter-dapr -o=jsonpath='{ .spec.ports[0].nodePort }')/"
echo "Dapr Dashboard  : http://"$(cat ~/work/server-ip)":$(kubectl get -n dapr-system service dapr-dashboard -o=jsonpath='{ .spec.ports[0].nodePort }')/"